In [0]:
# ==========================================
# STEP 4: GOLD LAYER - Feature Engineering (Kaggle Edition)
# ==========================================
from pyspark.sql.functions import col, max, min, count, avg, countDistinct, when, unix_timestamp

silver_sessions_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/silver/sessions/"
gold_features_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/gold/session_features/"

df_sessions = spark.read.format("delta").load(silver_sessions_s3_path)

print("Aggregating Kaggle events into session features...")
# Removed ip_address from the groupBy
df_grouped = df_sessions.groupBy("session_id", "user_id").agg(
    count("*").alias("event_count"),
    (unix_timestamp(max("event_time")) - unix_timestamp(min("event_time"))).alias("raw_duration_sec"),
    avg("gap_seconds").alias("avg_time_between_events"),
    countDistinct("product_id").alias("unique_products_viewed"),
    countDistinct("event_type").alias("unique_event_types")
)

df_features = df_grouped.withColumn(
    "session_duration_sec",
    when(col("raw_duration_sec") == 0, 1).otherwise(col("raw_duration_sec"))
).drop("raw_duration_sec")

df_final_features = df_features.withColumn(
    "click_rate_per_sec", col("event_count") / col("session_duration_sec")
).withColumn(
    "bounce_flag", when(col("event_count") == 1, 1).otherwise(0)
).withColumn(
    "action_diversity_score", col("unique_event_types") / col("event_count")
)

df_final_features.write.format("delta").mode("overwrite").save(gold_features_s3_path)
print("✅ Feature Engineering complete!")

In [0]:
# ==========================================
# STEP 5: GOLD LAYER - ML Anomaly Detection (FIXED)
# ==========================================
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import col

# 1. Define Paths
gold_features_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/gold/session_features/"
gold_scored_s3_path = "s3://useast1-nlsn-dbdata-userhome/users/govinda.jeswani/data_quality_check/gold/fraud_scores/"

print("Loading Gold features for ML training...")
df_features = spark.read.format("delta").load(gold_features_s3_path)

# 2. Assemble Features for Spark MLlib
feature_cols = ["event_count", "session_duration_sec", "click_rate_per_sec", "unique_products_viewed", "bounce_flag", "action_diversity_score"]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
df_assembled = assembler.transform(df_features)

# 3. Scale the Features (Crucial for distance-based algorithms like K-Means)
scaler = StandardScaler(inputCol="raw_features", outputCol="scaled_features", withStd=True, withMean=True)
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

print("Training Unsupervised ML Model (K-Means Anomaly Detection)...")
# 4. Train K-Means Model 
# FIX 1: We removed our custom predictionCol so Databricks MLflow autologging can use the default "prediction"
kmeans = KMeans(featuresCol="scaled_features", k=2, seed=42)
model = kmeans.fit(df_scaled)

# 5. Score the Sessions
df_scored = model.transform(df_scaled)

# 6. Identify which cluster is the "Anomaly" cluster
# FIX 2: We use pure PySpark to count the clusters and find the one with the smallest population (avoiding Python's min)
anomaly_cluster_row = df_scored.groupBy("prediction").count().orderBy("count").first()
anomaly_cluster = anomaly_cluster_row["prediction"]

print(f"Model identified Cluster {anomaly_cluster} as the Anomaly (Bot) cluster.")

# 7. Label the data
df_final_scores = df_scored.withColumn(
    "is_bot_anomaly", 
    (col("prediction") == anomaly_cluster).cast("boolean")
).drop("raw_features", "scaled_features", "prediction")

# 8. Write to Final Gold Scored Table
print(f"Writing final scored data to {gold_scored_s3_path}...")
df_final_scores.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_scored_s3_path)

print("✅ ML Scoring complete! The pipeline is fully functional end-to-end.")

# Let's see if it caught our bot!
print("\n--- ML Model Results: Flagged Bot Sessions ---")
display(
    df_final_scores.filter(col("is_bot_anomaly") == True)
    .select("user_id", "session_id", "event_count", "click_rate_per_sec", "is_bot_anomaly")
)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

print("--- Top 5 Most Aggressive Bots Caught by the ML Model ---")
display(
    df_final_scores.filter(col("is_bot_anomaly") == True)
    .orderBy(col("click_rate_per_sec").desc())
    .select("user_id", "session_id", "event_count", "session_duration_sec", "click_rate_per_sec", "unique_products_viewed")
    .limit(5)
)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Take a 1% random sample of our Gold table to safely plot in the browser
print("Sampling data for visualization...")
sample_df = df_final_scores.sample(fraction=0.01, seed=42).toPandas()

# 2. Graph 1: The Scatter Plot (Behavioral Separation)
plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=sample_df, 
    x="session_duration_sec", 
    y="event_count", 
    hue="is_bot_anomaly", 
    palette={False: "#3498db", True: "#e74c3c"}, # Blue for normal, Red for bots
    alpha=0.6,
    edgecolor=None
)
plt.title("Bot Detection: Session Duration vs. Event Count", fontsize=16, fontweight='bold')
plt.xlabel("Session Duration (Seconds)", fontsize=12)
plt.ylabel("Total Clicks/Events in Session", fontsize=12)
plt.legend(title="Is Bot?", labels=["Normal Traffic", "Flagged Bot"])
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

# 3. Graph 2: The Distribution of Click Rates
plt.figure(figsize=(12, 6))
sns.kdeplot(
    data=sample_df, 
    x="click_rate_per_sec", 
    hue="is_bot_anomaly", 
    fill=True, 
    common_norm=False, 
    palette={False: "#3498db", True: "#e74c3c"}
)
plt.title("Distribution of Click Rates (Bots vs. Humans)", fontsize=16, fontweight='bold')
plt.xlabel("Clicks Per Second", fontsize=12)
plt.ylabel("Density", fontsize=12)
# Limit the X-axis so the graph is readable (ignoring extreme massive outliers)
plt.xlim(-0.5, 5) 
plt.show()